# 07 · Decline stability check (US-06, CHUNKY-130)

Added by barissaka for CHUNKY-130 (US-06). The answer model does not always give the same reply to the same question: on 24 Nov 2026 the in-scope question S25 was answered in one run and declined in the next, with the same prompt. A single run therefore cannot show that in-scope questions are never declined.

This notebook answers the 10 in-scope development questions (`docs/eval/trial_in_scope_10.csv`) several times with each prompt and counts how often each one is declined or gets the no-advice notice. Wanted: 0 in every cell.

LLM calls: 10 questions x REPEATS x number of prompts (60 with the values below).

In [ ]:
import csv
import json
from datetime import date

from chunky.answer import answer
from chunky.index import ROOT, Embedder, load_config, load_index

PROMPTS = ["answer_v2", "answer_v3"]   # prompt files in prompts/ to compare
REPEATS = 3                            # how many times each question is asked with each prompt

cfg = load_config()
emb = cfg["embedding"]
embedder = Embedder(emb["model"], emb["revision"], emb["max_tokens"])
index = load_index()

with open(ROOT / "docs/eval/trial_in_scope_10.csv", encoding="utf-8-sig", newline="") as f:
    questions = [(q["Question ID"], q["Question"]) for q in csv.DictReader(f)]
print(len(questions), "in-scope questions |", PROMPTS, "|", REPEATS, "repeats")

In [ ]:
out = ROOT / "docs/eval/runs" / f"{date.today()}_decline_stability.jsonl"
out.parent.mkdir(parents=True, exist_ok=True)
rows = []
with open(out, "w", encoding="utf-8") as f:
    for prompt in PROMPTS:
        cfg["generation"]["prompt"] = prompt
        for repeat in range(1, REPEATS + 1):
            for qid, text in questions:
                r = answer(text, embedder, index, cfg)
                r.update(question_id=qid, repeat=repeat)
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
                rows.append(r)
            done = [r for r in rows if r["prompt"] == prompt and r["repeat"] == repeat]
            print(f"{prompt} run {repeat}: declined {sum(r['declined'] for r in done)}/{len(done)}, "
                  f"notice {sum(r['advice_notice'] for r in done)}/{len(done)}")
print("saved:", out)

In [ ]:
print(f"{'question':9}" + "".join(f"{p + ' declined':>20}{p + ' notice':>18}" for p in PROMPTS))
for qid, _ in questions:
    line = f"{qid:9}"
    for p in PROMPTS:
        mine = [r for r in rows if r["prompt"] == p and r["question_id"] == qid]
        line += f"{sum(r['declined'] for r in mine):>18}/{len(mine)}{sum(r['advice_notice'] for r in mine):>16}/{len(mine)}"
    print(line)
for p in PROMPTS:
    mine = [r for r in rows if r["prompt"] == p]
    print(f"{p}: declined {sum(r['declined'] for r in mine)}/{len(mine)} answers, "
          f"notice {sum(r['advice_notice'] for r in mine)}/{len(mine)} answers (wanted: 0 and 0)")